In [ ]:
import os
from PIL import Image
import numpy as np

In [ ]:
#Get List of Images
folder_path = 'Path to images and masks'
output_path = 'Output path'
extension = '.jpg'
imagelist = [f for f in os.listdir(folder_path) if f.endswith(extension)]
print(len(imagelist))

In [ ]:
for i in imagelist:
    #Check mask exists
    hdr = i[:-13]
    ismask = os.path.exists(folder_path+hdr+"_mask.png")
    if ismask:
        img = Image.open(folder_path+i)
        msk = Image.open(folder_path+hdr+"_mask.png")
        resized_img = img.resize((224, 224))
        resized_msk = msk.resize((224, 224))
        resized_img.save(output_path+"Images/"+hdr+".jpg")
        resized_msk.save(output_path+"Masks/"+hdr+"_mask.png")

In [ ]:
def crimmins_iteration(img):
    # Working with a float copy to prevent overflow during intermediate steps
    img = img.astype(np.float32)
    rows, cols = img.shape
    
    # Define directions for 8-neighbor processing: (dr, dc)
    # N-S, E-W, NW-SE, NE-SW
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]

    for dr, dc in directions:
        # Shift images to get neighbors 'a' and 'c' for pixel 'b'
        # b is the current pixel, a is neighbor in -direction, c is neighbor in +direction
        a = np.roll(img, (dr, dc), axis=(0, 1))
        c = np.roll(img, (-dr, -dc), axis=(0, 1))
        b = img

        # Step 1: Dark pixel hulling (raising darker pixels)
        # if a >= b + 2 then b = b + 1
        mask1 = (a >= b + 2)
        b[mask1] += 1
        
        # if a > b and b <= c then b = b + 1
        mask2 = (a > b) & (b <= c)
        b[mask2] += 1
        
        # if c > b and b <= a then b = b + 1
        mask3 = (c > b) & (b <= a)
        b[mask3] += 1
        
        # if c >= b + 2 then b = b + 1
        mask4 = (c >= b + 2)
        b[mask4] += 1

        # Step 2: Light pixel hulling (lowering brighter pixels)
        # if a <= b - 2 then b = b - 1
        mask5 = (a <= b - 2)
        b[mask5] -= 1
        
        # if a < b and b >= c then b = b - 1
        mask6 = (a < b) & (b >= c)
        b[mask6] -= 1
        
        # if c < b and b >= a then b = b - 1
        mask7 = (c < b) & (b >= a)
        b[mask7] -= 1
        
        # if c <= b - 2 then b = b - 1
        mask8 = (c <= b - 2)
        b[mask8] -= 1

        img = b
        
    return np.clip(img, 0, 255).astype(np.uint8)

def apply_crimmins(img, iterations=1):
    for _ in range(iterations):
        img = crimmins_iteration(img)
    return img

In [ ]:
for i in os.listdir("Path to images and masks"):
    oimg = Image.open("Path to images and masks"+i)
    print(np.array(oimg).shape)
    cimg = apply_crimmins(np.array(oimg),1)
    cimg.save("Path to output"+i)